# Reject stale evidence after source revisions

A quote can still exist after a document changes its meaning. This notebook shows an application-owned revision gate around evidence extracted with the Responses API. It separates three questions: does the quote exist, did the source change, and does the source actually support the claim?

**Learning outcome:** retain the exact retrieval snapshot used for extraction, attach its content digest in application code, and require re-review when the current source bytes differ. No model-generated hash is trusted.

All workshop data is fictional. This is an independent contribution draft, not an accepted or official OpenAI publication. Author: Sophia Cole (QueenFi703).

## Prerequisites and cost
Python 3.10+ is sufficient for the offline utilities and tests. Open this notebook with its neighboring Python files available in the same directory. Run all cells without a key: the default is a deterministic mock, not an API response. For notebook execution install nbformat, nbclient, and ipykernel. Optional live extraction needs openai and pydantic; see requirements.txt.

Live extraction is disabled below. Changing LIVE to True sends the synthetic source text to OpenAI and may incur API charges. Set OPENAI_API_KEY in your environment, not in the notebook. The configurable model defaults to gpt-6-astra; access depends on your account. Output is capped at 512 tokens, including reasoning tokens, so a small limit may produce incomplete output. There are no automatic retries. store=False does not by itself promise zero retention under every API policy.

In [ ]:
from revision_gate import Source, snapshot, bind_finding, revalidate
from live_extraction import extract_candidate

LIVE = False  # Explicit opt-in only; leave False for offline review and CI.
MODEL = "gpt-6-astra"
retrieved = snapshot([
    Source("workshop", "The fictional workshop opens at 10:00. Doors close at 17:00."),
    Source("seating", "The fictional workshop has twelve seats."),
])
candidate = extract_candidate(retrieved, live=LIVE, model=MODEL)
bound = bind_finding(candidate, retrieved)
print("Mode:", "live" if LIVE else "synthetic mock")
print("Initial check:", revalidate(bound, retrieved))


Mode: synthetic mock
Initial check: snapshot_matches_not_semantically_verified


## 1. Bind to the snapshot, not to a model assertion

The application freezes its retrieval mapping before the request. Source objects are immutable, and duplicate IDs are rejected. The model supplies only claim, source_id, and quote. bind_finding checks the ID and exact substring, then computes the digest from the retained source text. Extra fields, including a model-supplied digest, are rejected.

The reusable implementation is in revision_gate.py. Digests use SHA-256 over UTF-8 text with no normalization. Different Unicode compositions or whitespace will trigger a change, intentionally. For real pipelines, choose and version canonicalization at ingestion rather than silently changing it at validation time.

In [ ]:
assert bound.source_digest == retrieved[bound.source_id].digest
try:
    bind_finding({**candidate, "source_digest": "model-supplied-value"}, retrieved)
except ValueError:
    print("Rejected: the model cannot choose the stored digest.")
else:
    raise AssertionError("Unexpected digest accepted")


Rejected: the model cannot choose the stored digest.


## 2. Demonstrate the failure an exact quote check misses

The revised source appends a cancellation. The old quote still appears verbatim, so an existence-only gate passes. The source-revision gate instead reports source_changed. This is a conservative invalidation policy: even an irrelevant edit triggers re-review. It does not try to infer whether the edit changes the claim's meaning.

In [ ]:
current = snapshot([
    Source("workshop", retrieved["workshop"].text + " Update: today the workshop is canceled."),
    retrieved["seating"],
])
print("Old quote still exists:", bound.quote in current[bound.source_id].text)
print("Revision check:", revalidate(bound, current))
assert revalidate(bound, current) == "source_changed"


Old quote still exists: True
Revision check: source_changed


## 3. A passing digest is not a truth verdict

An unchanged snapshot can contain an exact quote paired with a false claim. The deliberately verbose passing status below keeps semantic support visibly unverified. Route matching snapshots to a separate support evaluation or human review; do not call them safe, accurate, approved, or current in the real world solely because this gate passes.

In [ ]:
unsupported = bind_finding({
    "claim": "The workshop opens at midnight.",
    "source_id": "workshop", "quote": "opens at 10:00",
}, retrieved)
print(revalidate(unsupported, retrieved))
assert revalidate(unsupported, retrieved) == "snapshot_matches_not_semantically_verified"


snapshot_matches_not_semantically_verified


## 4. Run the regression matrix

The tests cover unchanged evidence, changed surrounding context with surviving quotes, changed or removed sources, fabricated/misattributed/empty quotes, attempted model-supplied digests, duplicate IDs, immutable snapshots, Unicode changes, malformed fields, and the explicit false-claim counterexample. They test deterministic application behavior, not model quality.

In [ ]:
import unittest
from test_revision_gate import RevisionGateTests
suite = unittest.defaultTestLoader.loadTestsFromTestCase(RevisionGateTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()


................
----------------------------------------------------------------------
Ran 16 tests in 0.001s

OK


## Integration boundaries

- Capture the exact source snapshot sent to the model, not a later re-fetch. Associate every result with that request's snapshot; never reuse evidence after a failed extraction.
- Re-fetch or consult your authoritative document store before the final check. This demo only compares local synthetic snapshots; it does not discover remote updates.
- Check and consume a version atomically, or use conditional writes/version preconditions. A source can change after validation; this notebook does not solve that race.
- A digest is not a signature, a trusted timestamp, or proof of authentic origin. Both the stored binding and current corpus must be controlled by the application. An attacker who can replace both can defeat this check.
- Deleting or revoking access to a source should invalidate it. This example reports missing sources but does not implement access control or per-user authorization.
- Re-run retrieval/extraction and support review when source_changed occurs. Do not silently attach a new digest to old evidence, auto-approve an external action, or override an existing approval system.
- Matching source bytes prove neither completeness nor semantic entailment, and do not prevent prompt injection. Production pipelines still need source trust, authorization, support evaluation, and independent human approval where applicable.

## Relationship to existing work

[PR #2880](https://github.com/openai/openai-cookbook/pull/2880) already proposes an exact-quote citation gate. This example intentionally adds source-revision invalidation and application-owned snapshot binding rather than claiming a new quote matcher. [Citation formatting guidance](https://developers.openai.com/api/docs/guides/citation-formatting) covers preparing and rendering citations. Neither is a substitute for support checking.

API shape checked against the [Structured Outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs) and [Responses reference](https://developers.openai.com/api/reference/resources/responses/methods/create) on 2026-10-23. The live branch has not been executed for this draft. No API calls or credentials were used during preparation.